<a href="https://colab.research.google.com/github/mikailachmad/nlp-trio-bertiga/blob/main/src/scrapping/Scrappings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install requests beautifulsoup4 pandas

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [3]:
import requests

url = "https://www.cnbcindonesia.com/tech/20260914160141-37-767770/krisis-baru-hantam-amerika-trump-blak-blakan-bilang-begini"

response = requests.get(url)

print(response.status_code)
print(response.text[:500])

200
<!DOCTYPE html>
<html lang="id" class="scroll-smooth" data-critters-container>

    <head>
    <link rel="preconnect" href="https://analytics.google.com">
    <link href="https://analytics.google.com" rel="dns-prefetch">

    <link rel="preconnect" href="https://www.google-analytics.com">
    <link href="https://www.google-analytics.com" rel="dns-prefetch">

    <link rel="preconnect" href="https://www.googletagmanager.com">
    <link href="https://www.googletagmanager.com" rel="dns-prefetch">




In [4]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "html.parser")

In [5]:
print(soup.find("h1"))

<h1 class="mb-4 text-32 font-extrabold detail-title">
            Krisis Baru Hantam Amerika, Trump Blak-Blakan Bilang Begini            </h1>


In [6]:
paragraphs = soup.select(".article-content p")

for p in paragraphs:
    print(p.get_text(strip=True))

In [7]:
body = " ".join(
    p.get_text(" ", strip=True)
    for p in paragraphs
)

print(body)

In [8]:
import json

scripts = soup.find_all(
    "script",
    type="application/ld+json"
)

for script in scripts:
    try:
        data = json.loads(script.string)
        print(data)
    except:
        pass

{'@context': 'https://schema.org', '@type': 'BreadcrumbList', 'itemListElement': [{'@type': 'ListItem', 'position': 1, 'name': 'CNBC Indonesia', 'item': 'https://www.cnbcindonesia.com/'}, {'@type': 'ListItem', 'position': 2, 'name': 'Tech', 'item': 'https://www.cnbcindonesia.com/tech'}, {'@type': 'ListItem', 'position': 3, 'name': 'Berita Tech'}]}
{'@context': 'https://schema.org', '@type': 'WebPage', 'headline': 'Krisis Baru Hantam Amerika, Trump Blak-Blakan Bilang Begini', 'url': 'https://www.cnbcindonesia.com/tech/20260914160141-37-767770/krisis-baru-hantam-amerika-trump-blak-blakan-bilang-begini', 'datePublished': '2026-09-15T07:40:00+07:00', 'image': 'https://awsimages.detik.net.id/visual/2026/08/27/usa-trump-1787826019749_169.jpeg?w=1200', 'thumbnailUrl': 'https://awsimages.detik.net.id/visual/2026/08/27/usa-trump-1787826019749_169.jpeg?w=1200'}
{'@context': 'https://schema.org', '@type': 'NewsArticle', 'mainEntityOfPage': {'@type': 'WebPage', '@id': 'https://www.cnbcindonesia.co

In [9]:
headline = data.get("headline")
date_published = data.get("datePublished")

# Scrap 1 artikel

In [10]:
import requests
from bs4 import BeautifulSoup
import json


def scrape_article(url):
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=20
    )

    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    # -------------------------
    # Metadata dari JSON-LD
    # -------------------------
    headline = None
    date_published = None
    author = None

    scripts = soup.find_all(
        "script",
        type="application/ld+json"
    )

    for script in scripts:
        try:
            data = json.loads(script.string)

            if isinstance(data, dict):
                if not headline:
                    headline = data.get("headline")

                if not date_published:
                    date_published = data.get("datePublished")

        except (json.JSONDecodeError, TypeError):
            continue

    # -------------------------
    # Judul fallback
    # -------------------------
    if not headline:
        h1 = soup.find("h1")

        if h1:
            headline = h1.get_text(
                " ",
                strip=True
            )

    # -------------------------
    # Isi artikel
    # -------------------------
    paragraphs = soup.select(
        "article p"
    )

    body = " ".join(
        p.get_text(" ", strip=True)
        for p in paragraphs
    )

    return {
        "url": url,
        "title": headline,
        "date_published": date_published,
        "body": body
    }

In [11]:
data = scrape_article(url)

print(data)

{'url': 'https://www.cnbcindonesia.com/tech/20260914160141-37-767770/krisis-baru-hantam-amerika-trump-blak-blakan-bilang-begini', 'title': 'Krisis Baru Hantam Amerika, Trump Blak-Blakan Bilang Begini', 'date_published': '2026-09-15T07:40:00+07:00', 'body': 'Jakarta, CNBC Indonesia - Panggung politik\xa0Amerika Serikat (AS) mendadak dihantam krisis, menyusul benturan antara alarm bahaya kecerdasan buatan (Artificial Intelligence/AI) dan ambisi hegemonik Gedung Putih. Di satu sisi, para bos raksasa teknologi\xa0menggaungkan\xa0pentingnya rem darurat peningkatan AI demi mencegah \'petaka\' besar yang bisa berujung pada kepunahan umat manusia. Di sisi lain, Presiden AS Donald Trump justru kukuh menolak mengerem peningkatan AI karena bertekad memenangi perlombaan teknologi melawan China dengan cara apa pun. Tekanan publik memuncak setelah gelombang pengunduran diri peneliti internal Anthropic yang membunyikan alarm mengerikan. Ia secara lantang menyebut AI yang melaju tanpa kendali dapat me

# Article Discovery - Google News RSS

Bagian ini digunakan untuk menemukan banyak URL artikel CNBC secara otomatis. Google News RSS digunakan sebagai **discovery**, sedangkan isi artikel diambil langsung dari CNBC.


In [12]:
!pip install feedparser tqdm lxml


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 1.7 MB/s eta 0:00:00


In [13]:
import feedparser
import time
import random
from urllib.parse import quote, urlparse
from datetime import date, timedelta
from tqdm.auto import tqdm


In [14]:
# Periode sesuai tugas
START_DATE = date(2021, 9, 1)
END_DATE = date(2026, 9, 1)

# Query dibuat cukup luas. Filtering NLP dilakukan setelah data terkumpul.
SEARCH_QUERIES = [
    "site:cnbc.com geopolitics",
    "site:cnbc.com geopolitical",
    "site:cnbc.com war",
    "site:cnbc.com conflict",
    "site:cnbc.com Ukraine",
    "site:cnbc.com Russia",
    "site:cnbc.com China",
    "site:cnbc.com Israel",
    "site:cnbc.com Gaza",
    "site:cnbc.com Iran",
    "site:cnbc.com Middle East",
    "site:cnbc.com United States",
    "site:cnbc.com trade war",
    "site:cnbc.com tariffs",
    "site:cnbc.com dollar",
    "site:cnbc.com USD",
    "site:cnbc.com currency",
    "site:cnbc.com exchange rate",
    "site:cnbc.com Federal Reserve",
    "site:cnbc.com interest rates",
]


In [15]:
def generate_date_ranges(start_date, end_date, days=7):
    ranges = []
    current = start_date

    while current < end_date:
        next_date = min(current + timedelta(days=days), end_date)
        ranges.append((current, next_date))
        current = next_date

    return ranges


def build_google_news_rss_url(query, start_date, end_date):
    date_query = (
        f"{query} "
        f"after:{start_date.strftime('%Y-%m-%d')} "
        f"before:{end_date.strftime('%Y-%m-%d')}"
    )

    return (
        "https://news.google.com/rss/search?"
        f"q={quote(date_query)}&hl=en-US&gl=US&ceid=US:en"
    )


In [16]:
def discover_from_rss(query, start_date, end_date):
    rss_url = build_google_news_rss_url(query, start_date, end_date)

    try:
        response = requests.get(
            rss_url,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=30
        )
        response.raise_for_status()

        feed = feedparser.parse(response.content)
        results = []

        for entry in feed.entries:
            source = entry.get("source", {})
            source_title = source.get("title", "") if isinstance(source, dict) else ""
            source_href = source.get("href", "") if isinstance(source, dict) else ""

            results.append({
                "title_rss": entry.get("title", ""),
                "rss_link": entry.get("link", ""),
                "published_rss": entry.get("published", ""),
                "source": source_title,
                "source_url": source_href,
                "query": query,
                "search_start": str(start_date),
                "search_end": str(end_date),
            })

        return results

    except Exception as e:
        print(f"RSS error | {query} | {start_date} - {end_date} | {e}")
        return []


def is_cnbc_url(url):
    if not url:
        return False

    domain = urlparse(url).netloc.lower()
    return domain == "cnbc.com" or domain.endswith(".cnbc.com")


## Jalankan discovery

**Untuk testing terlebih dahulu**, jalankan rentang kecil seperti 1 bulan. Setelah hasilnya benar, baru jalankan 5 tahun.


In [17]:
# TEST terlebih dahulu
TEST_START = date(2025, 1, 1)
TEST_END = date(2025, 2, 1)

test_discovered = []

for query in SEARCH_QUERIES[:3]:
    for start_date, end_date in generate_date_ranges(TEST_START, TEST_END, days=7):
        test_discovered.extend(
            discover_from_rss(query, start_date, end_date)
        )
        time.sleep(random.uniform(0.5, 1.0))

test_discovered_df = pd.DataFrame(test_discovered)

print(f"Total hasil test: {len(test_discovered_df):,}")
test_discovered_df.head(10)


Total hasil test: 324


,title_rss,rss_link,published_rss,source,source_url,query,search_start,search_end
0,U.S. interference in Greenland could open up i...,https://news.google.com/rss/articles/CBMizAFBV...,"Wed, 08 Jan 2025 08:00:00 GMT",CNBC,https://www.cnbc.com,site:cnbc.com geopolitics,2025-01-01,2025-01-08
1,Several commodities face headwinds in 2025 — b...,https://news.google.com/rss/articles/CBMie0FVX...,"Mon, 06 Jan 2025 08:00:00 GMT",CNBC,https://www.cnbc.com,site:cnbc.com geopolitics,2025-01-01,2025-01-08
2,Tech group urges White House to halt rule that...,https://news.google.com/rss/articles/CBMivgFBV...,"Tue, 07 Jan 2025 08:00:00 GMT",CNBC,https://www.cnbc.com,site:cnbc.com geopolitics,2025-01-01,2025-01-08
3,How AI regulation could shake out in 2025 - CNBC,https://news.google.com/rss/articles/CBMipwFBV...,"Mon, 06 Jan 2025 08:00:00 GMT",CNBC,https://www.cnbc.com,site:cnbc.com geopolitics,2025-01-01,2025-01-08
4,Oil prices up 2% on China optimism as investor...,https://news.google.com/rss/articles/CBMioAFBV...,"Wed, 01 Jan 2025 08:00:00 GMT",CNBC,https://www.cnbc.com,site:cnbc.com geopolitics,2025-01-01,2025-01-08
5,Manthey: European markets have priced in uncer...,https://news.google.com/rss/articles/CBMiyAFBV...,"Tue, 07 Jan 2025 08:00:00 GMT",CNBC,https://www.cnbc.com,site:cnbc.com geopolitics,2025-01-01,2025-01-08
6,Biden blocks U.S. Steel takeover by Japan's Ni...,https://news.google.com/rss/articles/CBMitgFBV...,"Fri, 03 Jan 2025 08:00:00 GMT",CNBC,https://www.cnbc.com,site:cnbc.com geopolitics,2025-01-01,2025-01-08
7,Why Trump's pursuit of Greenland could be chee...,https://news.google.com/rss/articles/CBMinwFBV...,"Wed, 08 Jan 2025 08:00:00 GMT",CNBC,https://www.cnbc.com,site:cnbc.com geopolitics,2025-01-01,2025-01-08
8,Trump is fixated on Greenland — a vast Arctic ...,https://news.google.com/rss/articles/CBMipgFBV...,"Tue, 14 Jan 2025 08:00:00 GMT",CNBC,https://www.cnbc.com,site:cnbc.com geopolitics,2025-01-08,2025-01-15
9,"Conflict, extreme weather and disinformation t...",https://news.google.com/rss/articles/CBMipAFBV...,"Wed, 15 Jan 2025 08:00:00 GMT",CNBC,https://www.cnbc.com,site:cnbc.com geopolitics,2025-01-08,2025-01-15


In [18]:
# Setelah test berhasil, jalankan discovery seluruh periode
all_discovered = []
date_ranges = generate_date_ranges(START_DATE, END_DATE, days=7)

print(f"Jumlah query: {len(SEARCH_QUERIES)}")
print(f"Jumlah date window: {len(date_ranges)}")
print(f"Perkiraan RSS request: {len(SEARCH_QUERIES) * len(date_ranges):,}")

for query in SEARCH_QUERIES:
    for start_date, end_date in tqdm(date_ranges, desc=query):
        all_discovered.extend(
            discover_from_rss(query, start_date, end_date)
        )
        time.sleep(random.uniform(0.5, 1.2))

discovered_df = pd.DataFrame(all_discovered)

print(f"Total hasil sebelum dedup: {len(discovered_df):,}")


Jumlah query: 20
Jumlah date window: 261
Perkiraan RSS request: 5,220


site:cnbc.com geopolitics:   0%|          | 0/261 [00:00<?, ?it/s]

site:cnbc.com geopolitical:   0%|          | 0/261 [00:00<?, ?it/s]

site:cnbc.com war:   0%|          | 0/261 [00:00<?, ?it/s]

site:cnbc.com conflict:   0%|          | 0/261 [00:00<?, ?it/s]

...

site:cnbc.com currency:   0%|          | 0/261 [00:00<?, ?it/s]

site:cnbc.com exchange rate:   0%|          | 0/261 [00:00<?, ?it/s]

site:cnbc.com Federal Reserve:   0%|          | 0/261 [00:00<?, ?it/s]

site:cnbc.com interest rates:   0%|          | 0/261 [00:00<?, ?it/s]

Total hasil sebelum dedup: 205,258


In [19]:
discovered_df.to_csv("cnbc_discovered_raw.csv", index=False)

if not discovered_df.empty:
    # Utamakan source_url dari RSS untuk memastikan source adalah CNBC
    discovered_df["is_cnbc"] = discovered_df["source_url"].apply(is_cnbc_url)
    cnbc_discovered = discovered_df[discovered_df["is_cnbc"]].copy()

    # Dedup link
    cnbc_discovered = cnbc_discovered.drop_duplicates(
        subset=["rss_link"]
    ).copy()

    # Dedup judul
    cnbc_discovered["title_key"] = (
        cnbc_discovered["title_rss"]
        .fillna("")
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    cnbc_discovered = cnbc_discovered.drop_duplicates(
        subset=["title_key"]
    ).reset_index(drop=True)
else:
    cnbc_discovered = discovered_df.copy()

print(f"Unique candidate CNBC articles: {len(cnbc_discovered):,}")
cnbc_discovered.head(10)


Unique candidate CNBC articles: 67,280


,title_rss,rss_link,published_rss,source,source_url,query,search_start,search_end,is_cnbc,title_key
0,China's health-care sector could be Beijing's ...,https://news.google.com/rss/articles/CBMiqwFBV...,"Wed, 01 Sep 2021 07:00:00 GMT",CNBC,https://www.cnbc.com,site:cnbc.com geopolitics,2021-09-01,2021-09-08,True,china's health-care sector could be beijing's ...
1,"Biden, Xi discuss avoiding confrontation in se...",https://news.google.com/rss/articles/CBMijgFBV...,"Fri, 10 Sep 2021 07:00:00 GMT",CNBC,https://www.cnbc.com,site:cnbc.com geopolitics,2021-09-08,2021-09-15,True,"biden, xi discuss avoiding confrontation in se..."
2,EU chief calls for more military independence ...,https://news.google.com/rss/articles/CBMinAFBV...,"Wed, 15 Sep 2021 07:00:00 GMT",CNBC,https://www.cnbc.com,site:cnbc.com geopolitics,2021-09-08,2021-09-15,True,eu chief calls for more military independence ...
3,Op-ed: Will China's President Xi’s big bet pay...,https://news.google.com/rss/articles/CBMijAFBV...,"Sun, 19 Sep 2021 07:00:00 GMT",CNBC,https://www.cnbc.com,site:cnbc.com geopolitics,2021-09-15,2021-09-22,True,op-ed: will china's president xi’s big bet pay...
4,"As Merkel prepares to leave office, many think...",https://news.google.com/rss/articles/CBMipAFBV...,"Thu, 16 Sep 2021 07:00:00 GMT",CNBC,https://www.cnbc.com,site:cnbc.com geopolitics,2021-09-15,2021-09-22,True,"as merkel prepares to leave office, many think..."
5,India's Modi meets Kamala Harris ahead of bila...,https://news.google.com/rss/articles/CBMiqAFBV...,"Fri, 24 Sep 2021 07:00:00 GMT",CNBC,https://www.cnbc.com,site:cnbc.com geopolitics,2021-09-22,2021-09-29,True,india's modi meets kamala harris ahead of bila...
6,Chinese loans leave developing countries with ...,https://news.google.com/rss/articles/CBMitwFBV...,"Thu, 30 Sep 2021 07:00:00 GMT",CNBC,https://www.cnbc.com,site:cnbc.com geopolitics,2021-09-29,2021-10-06,True,chinese loans leave developing countries with ...
7,The Taliban takeover of Afghanistan could resh...,https://news.google.com/rss/articles/CBMixwFBV...,"Tue, 05 Oct 2021 07:00:00 GMT",CNBC,https://www.cnbc.com,site:cnbc.com geopolitics,2021-09-29,2021-10-06,True,the taliban takeover of afghanistan could resh...
8,As Wall Street sees a Chinese stock market 'on...,https://news.google.com/rss/articles/CBMiogFBV...,"Wed, 06 Oct 2021 07:00:00 GMT",CNBC,https://www.cnbc.com,site:cnbc.com geopolitics,2021-09-29,2021-10-06,True,as wall street sees a chinese stock market 'on...
9,5 charts show Russia's economic highs and lows...,https://news.google.com/rss/articles/CBMijAFBV...,"Mon, 11 Oct 2021 07:00:00 GMT",CNBC,https://www.cnbc.com,site:cnbc.com geopolitics,2021-10-06,2021-10-13,True,5 charts show russia's economic highs and lows...


# Bulk Scraping Artikel CNBC

Setelah URL ditemukan, cell berikut mengambil halaman artikel CNBC menggunakan **HTTP request**, lalu melakukan **HTML/JSON-LD parsing**.


In [20]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

session = requests.Session()
session.headers.update(HEADERS)


In [21]:
def extract_jsonld(soup):
    items = []

    for script in soup.find_all("script", type="application/ld+json"):
        if not script.string:
            continue

        try:
            data = json.loads(script.string)

            if isinstance(data, list):
                items.extend(data)
            elif isinstance(data, dict):
                items.append(data)

        except (json.JSONDecodeError, TypeError):
            continue

    return items


def extract_article_metadata(soup):
    headline = None
    date_published = None
    author = None

    for item in extract_jsonld(soup):
        if not isinstance(item, dict):
            continue

        item_type = item.get("@type")

        if isinstance(item_type, list):
            is_article = any(
                t in ["NewsArticle", "Article"]
                for t in item_type
            )
        else:
            is_article = item_type in ["NewsArticle", "Article"]

        if is_article or "headline" in item:
            headline = headline or item.get("headline")
            date_published = date_published or item.get("datePublished")

            if not author:
                author_data = item.get("author")

                if isinstance(author_data, dict):
                    author = author_data.get("name")

                elif isinstance(author_data, list):
                    names = [
                        a.get("name")
                        for a in author_data
                        if isinstance(a, dict) and a.get("name")
                    ]
                    author = ", ".join(names) if names else None

    if not headline:
        h1 = soup.find("h1")
        if h1:
            headline = h1.get_text(" ", strip=True)

    return {
        "title": headline,
        "published_at": date_published,
        "author": author,
    }


def extract_article_body(soup):
    selectors = [
        ".article-content p",
        "article p",
        "[class*='ArticleBody'] p",
        "[class*='article-body'] p",
        "[class*='ArticleBodyContainer'] p",
    ]

    paragraphs = []

    for selector in selectors:
        found = soup.select(selector)

        if len(found) >= 3:
            paragraphs = found
            break

    if not paragraphs:
        paragraphs = soup.find_all("p")

    texts = []

    for p in paragraphs:
        text = p.get_text(" ", strip=True)

        if text:
            texts.append(text)

    # Hilangkan paragraf yang sama tanpa mengubah urutan
    texts = list(dict.fromkeys(texts))

    return "\n".join(texts)


In [22]:
def scrape_article(url):
    try:
        response = session.get(
            url,
            timeout=30,
            allow_redirects=True
        )
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        metadata = extract_article_metadata(soup)
        body = extract_article_body(soup)

        return {
            "url": response.url,
            "title": metadata["title"],
            "published_at": metadata["published_at"],
            "author": metadata["author"],
            "body": body,
            "status": "success",
            "error": None,
        }

    except Exception as e:
        return {
            "url": url,
            "title": None,
            "published_at": None,
            "author": None,
            "body": None,
            "status": "failed",
            "error": str(e),
        }


## Tes satu artikel hasil discovery


In [23]:
if len(cnbc_discovered) == 0:
    raise ValueError("Tidak ada URL CNBC hasil discovery. Cek hasil RSS terlebih dahulu.")

test_url = cnbc_discovered.iloc[0]["rss_link"]

print("Test URL:")
print(test_url)

test_article = scrape_article(test_url)
test_article


Test URL:
https://news.google.com/rss/articles/CBMiqwFBVV95cUxPb3NsTXcwMm9YX2dxRV9VcWMwQlZFZlRIZE5ZZmRPcmVjLTdWbnhpQ250alB2ZmRreEdqaGdNZmtoc05LVTgxaVFVbDlSNlJrc2ZuSXBxZ1JOaUdETFBUeHNaeGw0eUpSNXY0YTZRSXFlTkxfRXZJczdNcFB3LS1ZR2RWRzRPSnhHSlJfbmx5RW5McDlwWXpjdktXc3g4dGYyaENZR3l1VVJWWDDSAbABQVVfeXFMTkh6VWQ4LW1NMWZiMW1RNEY4eklVNjctRTdYcnpMbFZpeG1KYW43dG80X0hiY0o1YkFxcHVUbVZNUndsSkhlSFAwNTR2c0tNQkJwaG5IWEc4Ny1VekptcVRlLXJ0SUZnVV9ZZmNPZWVuLXU2ZllpSzNZMXRJSG9xak84NGtVOWQxbDFnYWp4RzNfd2ZURV9QSzlDRHd2NmF5enBmbGtrOGpfVG5yM3hPd0o?oc=5


{'url': 'https://news.google.com/rss/articles/CBMiqwFBVV95cUxPb3NsTXcwMm9YX2dxRV9VcWMwQlZFZlRIZE5ZZmRPcmVjLTdWbnhpQ250alB2ZmRreEdqaGdNZmtoc05LVTgxaVFVbDlSNlJrc2ZuSXBxZ1JOaUdETFBUeHNaeGw0eUpSNXY0YTZRSXFlTkxfRXZJczdNcFB3LS1ZR2RWRzRPSnhHSlJfbmx5RW5McDlwWXpjdktXc3g4dGYyaENZR3l1VVJWWDDSAbABQVVfeXFMTkh6VWQ4LW1NMWZiMW1RNEY4eklVNjctRTdYcnpMbFZpeG1KYW43dG80X0hiY0o1YkFxcHVUbVZNUndsSkhlSFAwNTR2c0tNQkJwaG5IWEc4Ny1VekptcVRlLXJ0SUZnVV9ZZmNPZWVuLXU2ZllpSzNZMXRJSG9xak84NGtVOWQxbDFnYWp4RzNfd2ZURV9QSzlDRHd2NmF5enBmbGtrOGpfVG5yM3hPd0o?oc=5&hl=en-US&gl=US&ceid=US:en',
 'title': None,
 'published_at': None,
 'author': None,
 'body': '',
 'status': 'success',
 'error': None}

## Bulk scraping dengan checkpoint

Hasil disimpan setiap 100 artikel. Kalau runtime terputus, file checkpoint yang sudah dibuat tetap tersedia.


In [ ]:
CHECKPOINT_FILE = "cnbc_articles_checkpoint.csv"
FINAL_FILE = "cnbc_articles_final.csv"
BATCH_SIZE = 100

results = []

for i, row in tqdm(
    cnbc_discovered.iterrows(),
    total=len(cnbc_discovered),
    desc="Scraping CNBC"
):
    result = scrape_article(row["rss_link"])

    result["discovery_title"] = row.get("title_rss", "")
    result["discovery_published"] = row.get("published_rss", "")
    result["discovery_query"] = row.get("query", "")

    results.append(result)

    if len(results) % BATCH_SIZE == 0:
        pd.DataFrame(results).to_csv(
            CHECKPOINT_FILE,
            index=False
        )

    # Beri jeda antar request
    time.sleep(random.uniform(1.0, 2.0))

articles_df = pd.DataFrame(results)
articles_df.to_csv(FINAL_FILE, index=False)

print(f"Total diproses: {len(articles_df):,}")
print(f"Success: {(articles_df['status'] == 'success').sum():,}")
print(f"Failed : {(articles_df['status'] == 'failed').sum():,}")


Scraping CNBC:   0%|          | 0/67280 [00:00<?, ?it/s]

In [ ]:
articles_df[[
    "url",
    "title",
    "published_at",
    "author",
    "status"
]].head(10)
